In [1]:
#import data
import pandas as pd
df=pd.read_csv("electricity_bill_dataset.csv")

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45345 entries, 0 to 45344
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Fan              45345 non-null  int64  
 1   Refrigerator     45345 non-null  float64
 2   AirConditioner   45345 non-null  float64
 3   Television       45345 non-null  float64
 4   Monitor          45345 non-null  float64
 5   MotorPump        45345 non-null  int64  
 6   Month            45345 non-null  int64  
 7   City             45345 non-null  str    
 8   Company          45345 non-null  str    
 9   MonthlyHours     45345 non-null  int64  
 10  TariffRate       45345 non-null  float64
 11  ElectricityBill  45345 non-null  float64
dtypes: float64(6), int64(4), str(2)
memory usage: 4.2 MB


In [3]:
df.shape

(45345, 12)

In [4]:
df.describe()

,Fan,Refrigerator,AirConditioner,Television,Monitor,MotorPump,Month,MonthlyHours,TariffRate,ElectricityBill
count,45345.000000,45345.000000,45345.000000,45345.000000,45345.000000,45345.0,45345.000000,45345.000000,45345.000000,45345.000000
mean,13.990694,21.705458,1.503959,12.502635,2.865057,0.0,6.488058,515.083207,8.369648,4311.771307
std,5.470816,1.672575,1.115482,5.756007,3.894933,0.0,3.443252,122.618017,0.576992,1073.886406
min,5.000000,17.000000,0.000000,3.000000,1.000000,0.0,1.000000,95.000000,7.400000,807.500000
25%,9.000000,22.000000,1.000000,7.000000,1.000000,0.0,3.000000,429.000000,7.900000,3556.800000
50%,14.000000,22.000000,2.000000,13.000000,1.000000,0.0,6.000000,515.000000,8.400000,4299.400000
75%,19.000000,23.000000,2.000000,17.000000,1.000000,0.0,9.000000,601.000000,8.900000,5038.800000
max,23.000000,23.000000,3.000000,22.000000,12.000000,0.0,12.000000,926.000000,9.300000,8286.300000


In [5]:
df.columns

Index(['Fan', 'Refrigerator', 'AirConditioner', 'Television', 'Monitor',
       'MotorPump', 'Month', 'City', 'Company', 'MonthlyHours', 'TariffRate',
       'ElectricityBill'],
      dtype='str')

In [6]:
#null value treatment
df.isna().sum()

Fan                0
Refrigerator       0
AirConditioner     0
Television         0
Monitor            0
MotorPump          0
Month              0
City               0
Company            0
MonthlyHours       0
TariffRate         0
ElectricityBill    0
dtype: int64

In [7]:
# feature engineering
df['total_appliance_load']=(df['Television']+df['Fan']+df['Refrigerator']+df['Monitor']+df['AirConditioner']+df['MotorPump'])


In [8]:
cols_to_drop=(['MotorPump','AirConditioner','Monitor','Refrigerator','Fan','Television'])
cols_to_drop.append('ElectricityBill')

In [9]:
#feature selection
x=df.drop(labels=cols_to_drop,axis=1)
y=df['ElectricityBill']

In [10]:
x

,Month,City,Company,MonthlyHours,TariffRate,total_appliance_load
0,10,Hyderabad,Tata Power Company Ltd.,384,8.4,48.0
1,5,Vadodara,NHPC,488,7.8,47.0
2,7,Shimla,Jyoti Structure,416,7.7,42.0
3,6,Mumbai,Power Grid Corp,475,9.2,54.0
4,2,Mumbai,Ratnagiri Gas and Power Pvt. Ltd. (RGPPL),457,9.2,48.0
...,...,...,...,...,...,...
45340,9,Ahmedabad,Maha Transco – Maharashtra State Electricity T...,764,7.9,66.0
45341,2,New Delhi,Orient Green,572,8.5,66.0
45342,1,New Delhi,GE T&D India Limited,609,8.5,67.0
45343,12,Ratnagiri,TransRail Lighting,748,7.4,60.0


In [11]:
#preprocessing
cat=[]
num=[]
for i in x.columns:
  if x[i].dtype in ['int64', 'float64']:
    num.append(i)
  else:
    cat.append(i)

Xcat=x[cat]
Xnum=x[num]

from sklearn.preprocessing import LabelEncoder,StandardScaler
le=LabelEncoder()
ss=StandardScaler()

for i in Xcat.columns:
    Xcat[i]=le.fit_transform(Xcat[i])

Xnum=pd.DataFrame(ss.fit_transform(Xnum),columns=num)

x=Xcat.join(Xnum)

In [12]:
x

,City,Company,Month,MonthlyHours,TariffRate,total_appliance_load
0,5,27,1.019960,-1.069049,0.052604,-0.461003
1,15,13,-0.432171,-0.220877,-0.987284,-0.561928
2,14,8,0.148681,-0.808073,-1.160598,-1.066551
3,7,19,-0.141745,-0.326899,1.439121,0.144544
4,7,20,-1.303450,-0.473697,1.439121,-0.461003
...,...,...,...,...,...,...
45340,0,12,0.729534,2.030040,-0.813969,1.355638
45341,10,18,-1.303450,0.464185,0.225919,1.355638
45342,10,3,-1.593877,0.765938,0.225919,1.456563
45343,13,30,1.600813,1.899553,-1.680542,0.750091


In [13]:
#PICKLE
import pickle
with open('scaler.pkl','wb') as file1:  
    pickle.dump(ss,file1)

In [14]:
#train-test split 
from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=41)

In [15]:
xtrain.shape

(36276, 6)

In [16]:
xtest.shape

(9069, 6)

In [17]:
#model 1- linear regression
from sklearn.linear_model import LinearRegression
lr=LinearRegression()

model=lr.fit(xtrain,ytrain)

In [18]:
#model 2 - k nearest neighbors
from sklearn.neighbors import KNeighborsRegressor
knn=KNeighborsRegressor(n_neighbors=5)

model=knn.fit(xtrain,ytrain)

In [19]:
#model 3 - decision tree regressor
from sklearn.tree import DecisionTreeRegressor
dtr=DecisionTreeRegressor()

model=dtr.fit(xtrain,ytrain)

In [20]:
#model 4 - random forest
from sklearn.ensemble import RandomForestRegressor
rfr=RandomForestRegressor(n_estimators=20)

model=rfr.fit(xtrain,ytrain)

In [21]:
#model 5 - adaboost regressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import AdaBoostRegressor

lir=LinearRegression()
abr=AdaBoostRegressor(lir,n_estimators=25)

model=abr.fit(xtrain,ytrain)


In [22]:
#model 6 - support vector machine
from sklearn.svm import SVR
svr=SVR()
model=svr.fit(xtrain,ytrain)

In [23]:
#model evaluation
from sklearn.metrics import r2_score,mean_absolute_error
models = {
    "Linear Regression": lr,
    "Decision Tree": dtr,
    "Random Forest": rfr,
    "KNN": knn,
    "AdaBoost": abr,
    "SVR": svr
}

for name, model in models.items():
    y_pred = model.predict(xtest)
    print(name)
    print("R2 Score:", r2_score(ytest, y_pred))
    print("MAE:", mean_absolute_error(ytest, y_pred))
    print("-"*30)

Linear Regression
R2 Score: 0.9957025716989809
MAE: 48.69864384863685
------------------------------
Decision Tree
R2 Score: 0.9999436351213322
MAE: 1.5831844745838224
------------------------------
Random Forest
R2 Score: 0.9999816229712251
MAE: 1.4694249641638963
------------------------------
KNN
R2 Score: 0.946372028896908
MAE: 193.80557724115118
------------------------------
AdaBoost
R2 Score: 0.9956901662017336
MAE: 48.77556013657704
------------------------------
SVR
R2 Score: 0.2694516559146033
MAE: 737.8308943781324
------------------------------


In [24]:
from sklearn.ensemble import RandomForestRegressor
rfr=RandomForestRegressor(n_estimators=20)

model=rfr.fit(xtrain,ytrain)

trpred=model.predict(xtrain)
tspred=model.predict(xtest)

from sklearn.metrics import r2_score

trscore=r2_score(ytrain,trpred)
tsscore=r2_score(ytest,tspred)
print('Training Score:',trscore)
print('Testing Score:',tsscore)

Training Score: 0.999992435406452
Testing Score: 0.9999824976616818


In [25]:
#PICKLE 
with open('model.pkl', 'wb') as file2:
    pickle.dump(model,file2)

In [26]:
with open('model.pkl' , 'rb') as file4:  # rb stands for read binary
    m= pickle.load(file4)